# Day 10: End-to-End Orchestration, Observability & Verification

This notebook simulates a complete End-to-End run of a synthetic patient from raw HL7 JSON through the pipeline layers to a Gold metric.

In [ ]:
import duckdb
import pandas as pd
import json
import hashlib
from datetime import datetime

con = duckdb.connect(database=':memory:')
print("E2E Simulation Environment Ready.")

### Step 1: Raw to Bronze

In [ ]:
# Synthetic Raw HL7 FHIR JSON
raw_payload = {
    "resourceType": "Patient",
    "id": "pat-999",
    "identifier": [{"system": "SSN", "value": "111-22-3333"}],
    "name": [{"family": "Smith", "given": ["Alice"]}],
    "encounter": [
        {"id": "enc-1", "period": {"start": "2023-11-01", "end": "2023-11-05"}},
        {"id": "enc-2", "period": {"start": "2023-11-15", "end": "2023-11-18"}} # 10 day readmission
    ]
}

# Simulate Bronze Ingestion
bronze_df = pd.DataFrame([{
    "payload_id": "uuid-123",
    "source_system_id": "HL7_STREAM",
    "ingestion_timestamp": datetime.now(),
    "raw_payload_json": json.dumps(raw_payload)
}])

con.execute("CREATE TABLE bronze_table AS SELECT * FROM bronze_df")
print("Bronze Data:")
print(con.execute("SELECT payload_id, source_system_id, raw_payload_json FROM bronze_table").df())

### Step 2: Bronze to Silver (Parse, De-id, OMOP)

In [ ]:
# Simulate Parsing and De-identification logic
def process_to_silver(row):
    payload = json.loads(row['raw_payload_json'])
    ssn = payload['identifier'][0]['value']
    
    # FPE Tokenization
    tokenized_ssn = "TKN-" + hashlib.sha256(ssn.encode()).hexdigest()[:10]
    
    # Extract encounters for OMOP mapping
    encounters = []
    for enc in payload['encounter']:
        encounters.append({
            "person_id": tokenized_ssn,
            "encounter_id": enc['id'],
            "encounter_start_date": enc['period']['start'],
            "encounter_end_date": enc['period']['end']
        })
    return encounters

silver_records = []
for _, row in bronze_df.iterrows():
    silver_records.extend(process_to_silver(row))

silver_df = pd.DataFrame(silver_records)
con.execute("CREATE TABLE silver_encounter AS SELECT * FROM silver_df")
print("\nSilver Data (OMOP Encounters, De-identified):")
print(con.execute("SELECT * FROM silver_encounter").df())

### Step 3: Silver to Gold (Readmission Metric)

In [ ]:
# Simulate dbt Gold transformation
gold_query = """
CREATE TABLE gold_readmissions AS
SELECT
    e1.person_id,
    e1.encounter_id AS index_encounter_id,
    e1.encounter_end_date AS index_discharge_date,
    e2.encounter_id AS readmission_encounter_id,
    e2.encounter_start_date AS readmission_date,
    date_diff('day', CAST(e1.encounter_end_date AS DATE), CAST(e2.encounter_start_date AS DATE)) AS days_to_readmission
FROM silver_encounter e1
JOIN silver_encounter e2 
    ON e1.person_id = e2.person_id
    AND e1.encounter_id != e2.encounter_id
    AND CAST(e2.encounter_start_date AS DATE) > CAST(e1.encounter_end_date AS DATE)
    AND date_diff('day', CAST(e1.encounter_end_date AS DATE), CAST(e2.encounter_start_date AS DATE)) <= 30
"""
con.execute(gold_query)

print("\nGold Data (30-Day Readmission Metric computed from E2E flow):")
print(con.execute("SELECT * FROM gold_readmissions").df())